# 02 — Train the MLP

This notebook trains the MLP configuration used for the thesis and ISMRM analysis using the synthetic dataset generated in Notebook 01.

### Training configuration

- input: 131 normalised diffusion-weighted measurements;
- architecture: **131 → 512 → 512 → 256 → 128 → 9**;
- activation: ReLU;
- dropout: 0.05;
- optimiser: AdamW;
- initial learning rate: 3 × 10⁻⁴;
- weight decay: 10⁻⁵;
- batch size: 1024;
- gradient clipping: 1.0;
- maximum epochs: 180;
- early-stopping patience: 15;
- ReduceLROnPlateau factor: 0.5, patience: 5, minimum learning rate: 10⁻⁶;
- training seed: 42.

During training, each signal receives dynamically generated Rician noise. An SNR is independently sampled from \(U(30,70)\) for each signal whenever it is used.

Feature means and standard deviations are estimated from one deterministic noisy realisation of the training partition only (seed 4242). Validation uses one fixed variable-SNR realisation (seed 9876). The held-out test partition is not used during training or checkpoint selection.

In [ ]:
from pathlib import Path
import os
import gc
import json
import time
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score


cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

DATA_DIR = PROJECT_ROOT / "data" / "synthetic"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results" / "training"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

X_CLEAN_FILE = DATA_DIR / "X_clean.pt"
Y_FILE = DATA_DIR / "y_labels.pt"
SPLIT_FILE = DATA_DIR / "master_split_80train_10val_10test_seed42.npz"
METADATA_FILE = DATA_DIR / "metadata.pt"

for path in [X_CLEAN_FILE, Y_FILE, SPLIT_FILE, METADATA_FILE]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}\n"
            "Run 01_generate_synthetic_data.ipynb first."
        )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
SEED = 42

HIDDEN_DIMS = (512, 512, 256, 128)
DROPOUT = 0.05

BATCH_SIZE = 1024
INITIAL_LR = 3e-4
WEIGHT_DECAY = 1e-5
GRAD_CLIP = 1.0

MAX_EPOCHS = 180
EARLY_STOPPING_PATIENCE = 15
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 5
MIN_LR = 1e-6

RANDOM_SNR_LOW = 30.0
RANDOM_SNR_HIGH = 70.0

STANDARDISATION_NOISE_SEED = 4242
VALIDATION_NOISE_SEED = 9876

In [ ]:
y = torch.load(Y_FILE, map_location="cpu").long()

with np.load(SPLIT_FILE) as split:
    train_idx = split["train_idx"].astype(np.int64)
    val_idx = split["val_idx"].astype(np.int64)

train_idx_t = torch.from_numpy(train_idx)
val_idx_t = torch.from_numpy(val_idx)

X_clean_full = torch.load(X_CLEAN_FILE, map_location="cpu").float()

X_train_clean = X_clean_full.index_select(0, train_idx_t).contiguous()
X_val_clean = X_clean_full.index_select(0, val_idx_t).contiguous()

y_train = y.index_select(0, train_idx_t).contiguous()
y_val = y.index_select(0, val_idx_t).contiguous()

del X_clean_full
gc.collect()

In [ ]:
metadata = torch.load(METADATA_FILE, map_location="cpu", weights_only=False)

CLASS_NAMES = metadata["model_order"]

assert metadata["n_dw_retained"] == 131

In [ ]:
class FinalBrainModelMLP(nn.Module):
    def __init__(
        self,
        input_dim=131,
        num_classes=9,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    ):
        super().__init__()

        layers = []
        previous = input_dim

        for hidden in hidden_dims:
            layers.extend([
                nn.Linear(previous, hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            previous = hidden

        layers.append(nn.Linear(previous, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cpu_state_dict(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }

In [ ]:
def add_fresh_variable_rician_noise(x, generator):
    """Add Rician noise with one SNR sampled from 30–70 per signal."""

    snr = torch.empty(
        (x.shape[0], 1),
        device=x.device,
        dtype=x.dtype,
    )
    snr.uniform_(
        RANDOM_SNR_LOW,
        RANDOM_SNR_HIGH,
        generator=generator,
    )

    sigma = 1.0 / snr

    real_noise = torch.randn(
        x.shape,
        device=x.device,
        dtype=x.dtype,
        generator=generator,
    )
    imag_noise = torch.randn(
        x.shape,
        device=x.device,
        dtype=x.dtype,
        generator=generator,
    )

    return torch.sqrt(
        (x + sigma * real_noise) ** 2
        + (sigma * imag_noise) ** 2
    )


@torch.no_grad()
def estimate_random_snr_stats(
    X_clean,
    seed=STANDARDISATION_NOISE_SEED,
    batch_size=20_000,
):
    """Estimate feature mean and standard deviation from noisy training data."""

    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)

    feature_sum = torch.zeros(X_clean.shape[1], dtype=torch.float64)
    feature_sumsq = torch.zeros(X_clean.shape[1], dtype=torch.float64)
    n_total = 0

    for start in range(0, len(X_clean), batch_size):
        clean = X_clean[start:start + batch_size]
        noisy = add_fresh_variable_rician_noise(clean, generator).double()

        feature_sum += noisy.sum(dim=0)
        feature_sumsq += (noisy**2).sum(dim=0)
        n_total += len(clean)

    mean = feature_sum / n_total
    variance = feature_sumsq / n_total - mean**2
    std = torch.sqrt(torch.clamp(variance, min=1e-12))

    return mean.float(), std.float()


@torch.no_grad()
def create_fixed_random_snr_dataset(
    X_clean,
    seed=VALIDATION_NOISE_SEED,
    batch_size=20_000,
):
    """Create one fixed noisy realisation of the validation data."""

    result = torch.empty_like(X_clean)

    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)

    for start in range(0, len(X_clean), batch_size):
        end = min(start + batch_size, len(X_clean))
        result[start:end] = add_fresh_variable_rician_noise(
            X_clean[start:end],
            generator,
        )

    return result

In [ ]:
# Estimates feature statistics from the noisy training.
feature_mean, feature_std = estimate_random_snr_stats(
    X_train_clean,
    seed=STANDARDISATION_NOISE_SEED,
)

# Creates fixed noisy validation realisation.
X_val_random30_70 = create_fixed_random_snr_dataset(
    X_val_clean,
    seed=VALIDATION_NOISE_SEED,
)

In [ ]:
@torch.no_grad()
def evaluate_validation(model, X_raw, y_true, feature_mean, feature_std):
    model.eval()

    loader = DataLoader(
        TensorDataset(X_raw, y_true),
        batch_size=4096,
        shuffle=False,
    )

    mean_d = feature_mean.to(device).reshape(1, -1)
    std_d = feature_std.to(device).reshape(1, -1)

    criterion = nn.CrossEntropyLoss()

    total_loss = 0.0
    total_n = 0
    true_all = []
    pred_all = []

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        xb = (xb - mean_d) / std_d

        logits = model(xb)
        loss = criterion(logits, yb)

        n = len(yb)
        total_loss += loss.item() * n
        total_n += n

        pred = torch.argmax(logits, dim=1)
        true_all.append(yb.cpu().numpy())
        pred_all.append(pred.cpu().numpy())

    y_true_np = np.concatenate(true_all)
    y_pred_np = np.concatenate(pred_all)

    return {
        "loss": total_loss / total_n,
        "accuracy": accuracy_score(y_true_np, y_pred_np),
        "macro_f1": f1_score(
            y_true_np,
            y_pred_np,
            average="macro",
            zero_division=0,
        ),
    }

## Train the MLP

The next cell trains the MLP and may take substantial time.

The frozen model used in the work is already provided in `checkpoints/` together with its feature-standardisation statistics, so retraining is OPTIONAL to reproduce the later analyses.

In [ ]:
RETRAINED_DIR = RESULTS_DIR / "retrained_model"
RETRAINED_DIR.mkdir(parents=True, exist_ok=True)

STATE_FILE = RETRAINED_DIR / "MLP_FRESH_RANDOM_SNR30_70_state_dict.pt"
STATS_FILE = RETRAINED_DIR / "MLP_FRESH_RANDOM_SNR30_70_feature_standardisation.pt"
HISTORY_FILE = RETRAINED_DIR / "MLP_FRESH_RANDOM_SNR30_70_training_history.csv"


set_all_seeds(SEED)

model = FinalBrainModelMLP().to(device)
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=INITIAL_LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE,
    min_lr=MIN_LR,
)

loader_generator = torch.Generator(device="cpu")
loader_generator.manual_seed(SEED)

noise_generator = torch.Generator(device=device.type)
noise_generator.manual_seed(SEED + 1000)

train_loader = DataLoader(
    TensorDataset(X_train_clean, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
)

mean_d = feature_mean.to(device).reshape(1, -1)
std_d = feature_std.to(device).reshape(1, -1)

history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_macro_f1": [],
    "learning_rate": [],
}

best_val_loss = np.inf
best_epoch = None
best_state = None
no_improvement = 0


for epoch in range(1, MAX_EPOCHS + 1):
    model.train()

    running_loss = 0.0
    running_n = 0

    for clean_batch, labels in train_loader:
        clean_batch = clean_batch.to(device)
        labels = labels.to(device)

        # Add dynamic Rician noise before feature standardisation.
        noisy_batch = add_fresh_variable_rician_noise(
            clean_batch,
            generator=noise_generator,
        )
        noisy_batch = (noisy_batch - mean_d) / std_d

        optimizer.zero_grad(set_to_none=True)

        logits = model(noisy_batch)
        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        running_loss += loss.item() * len(labels)
        running_n += len(labels)

    train_loss = running_loss / running_n

    val_result = evaluate_validation(
        model,
        X_val_random30_70,
        y_val,
        feature_mean,
        feature_std,
    )

    val_loss = val_result["loss"]
    current_lr = optimizer.param_groups[0]["lr"]

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_result["accuracy"])
    history["val_macro_f1"].append(val_result["macro_f1"])
    history["learning_rate"].append(current_lr)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = cpu_state_dict(model)
        no_improvement = 0
    else:
        no_improvement += 1

    scheduler.step(val_loss)

    print(
        f"Epoch {epoch:03d} | "
        f"train loss {train_loss:.5f} | "
        f"val loss {val_loss:.5f} | "
        f"val acc {val_result['accuracy']:.4f} | "
        f"val F1 {val_result['macro_f1']:.4f} | "
        f"lr {current_lr:.2e}"
    )

    if no_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping.")
        break


# Restore the checkpoint with the lowest validation loss.
model.load_state_dict(best_state)

final_val = evaluate_validation(
    model,
    X_val_random30_70,
    y_val,
    feature_mean,
    feature_std,
)

torch.save(cpu_state_dict(model), STATE_FILE)

torch.save(
    {
        "feature_mean": feature_mean.cpu(),
        "feature_std": feature_std.cpu(),
    },
    STATS_FILE,
)

pd.DataFrame(history).to_csv(HISTORY_FILE, index=False)

print("Best epoch:", best_epoch)
print("Validation accuracy:", f"{final_val['accuracy']:.5f}")
print("Validation macro-F1:", f"{final_val['macro_f1']:.5f}")
print("Retrained model saved to:", RETRAINED_DIR)

### End of Notebook 02

Notebook 03 evaluates the frozen thesis checkpoint on the held-out data.